# Transformaciones satelitales de 1 a 15 características — 60/20/20

Este notebook genera exactamente **60 CSV** para Camila:

- `PCA_01_caracteristicas.csv` hasta `PCA_15_caracteristicas.csv`
- `ICA_01_caracteristicas.csv` hasta `ICA_15_caracteristicas.csv`
- `AE_lineal_01_caracteristicas.csv` hasta `AE_lineal_15_caracteristicas.csv`
- `AE_no_lineal_01_caracteristicas.csv` hasta `AE_no_lineal_15_caracteristicas.csv`

Cada CSV reúne los cinco folds. En cada fold contiene:

- `train`: aproximadamente 60 %
- `validation`: aproximadamente 20 %
- `test`: aproximadamente 20 %

`validation` y `test` tienen exactamente el mismo número de muestras dentro de cada fold.

## Corrección crítica

El campo `split` original del NPZ se conserva como **`split_original`**. Nunca puede sobrescribir la nueva columna `split`, que contiene únicamente:

```text
train
validation
test
```

## Flujo sin aprendizaje compartido

```text
z64
→ división train/validation/test
→ StandardScaler ajustado solo con train
→ PCA / ICA / AE ajustado solo con train
→ MinMaxScaler ajustado solo con train
→ transformación de validation y test sin reajustar
→ CSV
```

Los autoencoders actualizan sus pesos únicamente con train. Validation se usa solo para seleccionar el mejor checkpoint.

## Aceleraciones incluidas

- El `StandardScaler` se calcula una sola vez por fold.
- PCA se ajusta una sola vez con 15 componentes por fold y se reutilizan sus primeras `k` componentes.
- Los tensores de los autoencoders permanecen en GPU durante el entrenamiento.
- Batch grande configurable.
- Early stopping.
- Escritura atómica de CSV.
- Reanudación automática: un CSV ya completo y válido se omite.

In [1]:
# ============================================================
# Celda 1 — Imports y configuración
# ============================================================
import os
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.decomposition import PCA, FastICA
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------
# Reproducibilidad
# -----------------------------
SEED = 99
N_SPLITS = 5
MIN_FEATURES = 1
MAX_FEATURES = 15
SCALE = np.pi / 2

# -----------------------------
# Autoencoders
# -----------------------------
AE_MAX_EPOCHS = 100
AE_BATCH_SIZE = 1024
AE_LR = 1e-3
AE_WEIGHT_DECAY = 1e-5
AE_HIDDEN = 32
AE_PATIENCE = 12
AE_MIN_DELTA = 1e-7

# Guardar pesos es opcional. False reduce escritura en disco.
SAVE_AE_WEIGHTS = False

# Reanuda y omite CSV que ya estén completos y verificados.
RESUME = True

# None usa todo el dataset. Para una prueba rápida: 2000.
SAMPLE_SIZE = None

DATA_PATH = r"C:\Users\lapic\datasets\z64_dimensionality_reduction_5fold\distr_sol_pv_segm_embeddings_v2"

NOMBRE_TRABAJO = (
    "Comparacion_de_transformaciones_de_1_a_15_caracteristicas_"
    "con_validacion_60_20_20"
)

# Esta carpeta contendrá solamente los 60 CSV finales.
OUTPUT_DIR = Path(NOMBRE_TRABAJO)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Información auxiliar separada.
AUX_DIR = Path(f"{NOMBRE_TRABAJO}_auxiliares")
TABLES_DIR = AUX_DIR / "tablas"
WEIGHTS_DIR = AUX_DIR / "pesos_autoencoders"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

CONFIG = {
    "seed": SEED,
    "n_splits": N_SPLITS,
    "division": "60/20/20 aproximada",
    "validation_equals_test": True,
    "min_features": MIN_FEATURES,
    "max_features": MAX_FEATURES,
    "final_range": [-float(SCALE), float(SCALE)],
    "ae_max_epochs": AE_MAX_EPOCHS,
    "ae_batch_size": AE_BATCH_SIZE,
    "ae_lr": AE_LR,
    "ae_weight_decay": AE_WEIGHT_DECAY,
    "ae_hidden": AE_HIDDEN,
    "ae_patience": AE_PATIENCE,
    "ae_min_delta": AE_MIN_DELTA,
    "save_ae_weights": SAVE_AE_WEIGHTS,
    "resume": RESUME,
    "sample_size": SAMPLE_SIZE,
    "data_path": DATA_PATH,
    "output_dir": str(OUTPUT_DIR),
    "device": str(DEVICE),
}

with (AUX_DIR / "config.json").open("w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

print("Dispositivo:", DEVICE)
print("Carpeta de salida:", OUTPUT_DIR.resolve())
print("Batch AE:", AE_BATCH_SIZE)
print("Reanudación:", RESUME)

Dispositivo: cuda
Carpeta de salida: C:\Users\lapic\QML5\Z64_Carpetas\Comparacion_de_transformaciones_de_1_a_15_caracteristicas_con_validacion_60_20_20
Batch AE: 1024
Reanudación: True


In [2]:
# ============================================================
# Celda 2 — Carga segura del NPZ
# ============================================================
RESERVED_COLUMNS = {"fold", "split", "sample_index", "label"}


def resolve_npz_path(path: str) -> Path:
    candidate = Path(path)
    if candidate.exists():
        return candidate

    if candidate.suffix.lower() != ".npz":
        candidate_npz = Path(str(candidate) + ".npz")
        if candidate_npz.exists():
            return candidate_npz

    raise FileNotFoundError(
        f"No se encontró el dataset en:\n{path}\n"
        "Revisa DATA_PATH en la Celda 1."
    )


def load_dataset(path: str, sample_size=None, seed: int = SEED):
    resolved = resolve_npz_path(path)

    with np.load(resolved, allow_pickle=True) as data:
        required = {"features", "label"}
        missing = required - set(data.files)
        if missing:
            raise KeyError(f"Faltan claves obligatorias en el NPZ: {sorted(missing)}")

        X = np.asarray(data["features"], dtype=np.float32)
        y = np.asarray(data["label"], dtype=np.int64).reshape(-1)

        if X.ndim != 2:
            raise ValueError(f"'features' debe ser 2D; se recibió {X.shape}.")
        if len(X) != len(y):
            raise ValueError("features y label no tienen la misma cantidad de filas.")
        if not np.isfinite(X).all():
            raise ValueError("features contiene NaN o valores infinitos.")

        original_index = np.arange(len(X), dtype=np.int64)

        metadata = {}

        if "image_id" in data.files:
            metadata["image_id"] = np.asarray(data["image_id"])

        if "crop_id" in data.files:
            metadata["crop_id"] = np.asarray(data["crop_id"])

        # CORRECCIÓN: nunca se guarda con el nombre reservado "split".
        if "split" in data.files:
            metadata["split_original"] = np.asarray(data["split"])

    # Muestreo estratificado opcional.
    if sample_size is not None:
        sample_size = int(sample_size)
        if sample_size <= 0:
            raise ValueError("SAMPLE_SIZE debe ser None o un entero positivo.")

        if sample_size < len(X):
            keep_idx, _ = train_test_split(
                np.arange(len(X)),
                train_size=sample_size,
                random_state=seed,
                stratify=y,
                shuffle=True,
            )
            keep_idx = np.sort(keep_idx)

            X = X[keep_idx]
            y = y[keep_idx]
            original_index = original_index[keep_idx]
            metadata = {
                key: np.asarray(values)[keep_idx]
                for key, values in metadata.items()
            }

    # Verificación de longitudes.
    for key, values in metadata.items():
        if len(values) != len(X):
            raise ValueError(
                f"El metadato '{key}' tiene {len(values)} filas y X tiene {len(X)}."
            )
        if key in RESERVED_COLUMNS:
            raise RuntimeError(
                f"El metadato '{key}' intenta usar una columna reservada."
            )

    return X, y, original_index, metadata, resolved


X_all, y_all, original_index_all, metadata_all, resolved_data_path = load_dataset(
    DATA_PATH,
    sample_size=SAMPLE_SIZE,
)

print("Dataset:", resolved_data_path)
print("X:", X_all.shape, X_all.dtype)
print("y:", y_all.shape, y_all.dtype)
print("Clases:", dict(zip(*np.unique(y_all, return_counts=True))))
print("Metadatos exportados:", list(metadata_all))
print("Nota: el split del NPZ se exporta como 'split_original'.")

Dataset: C:\Users\lapic\datasets\z64_dimensionality_reduction_5fold\distr_sol_pv_segm_embeddings_v2.npz
X: (20976, 64) float32
y: (20976,) int64
Clases: {0: 10293, 1: 10683}
Metadatos exportados: ['image_id', 'crop_id', 'split_original']
Nota: el split del NPZ se exporta como 'split_original'.


In [3]:
# ============================================================
# Celda 3 — Folds 60/20/20 y estandarización reutilizable
# ============================================================
outer_skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

FOLDS = []

for fold, (train_validation_idx, test_idx) in enumerate(
    outer_skf.split(X_all, y_all),
    start=1,
):
    # Se extraen exactamente tantas muestras para validation como hay en test.
    train_idx, validation_idx = train_test_split(
        train_validation_idx,
        test_size=len(test_idx),
        random_state=SEED + fold,
        stratify=y_all[train_validation_idx],
        shuffle=True,
    )

    train_idx = np.asarray(train_idx, dtype=np.int64)
    validation_idx = np.asarray(validation_idx, dtype=np.int64)
    test_idx = np.asarray(test_idx, dtype=np.int64)

    if len(validation_idx) != len(test_idx):
        raise RuntimeError(
            f"Fold {fold}: validation y test no tienen el mismo tamaño."
        )

    # Verificación de disjunción.
    if np.intersect1d(train_idx, validation_idx).size:
        raise RuntimeError(f"Fold {fold}: solapamiento train-validation.")
    if np.intersect1d(train_idx, test_idx).size:
        raise RuntimeError(f"Fold {fold}: solapamiento train-test.")
    if np.intersect1d(validation_idx, test_idx).size:
        raise RuntimeError(f"Fold {fold}: solapamiento validation-test.")

    all_idx = np.concatenate([train_idx, validation_idx, test_idx])
    if len(np.unique(all_idx)) != len(X_all):
        raise RuntimeError(f"Fold {fold}: la partición no cubre exactamente el dataset.")

    # El StandardScaler se ajusta una sola vez por fold, únicamente con train.
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_all[train_idx]).astype(np.float32)
    X_validation_std = scaler.transform(X_all[validation_idx]).astype(np.float32)
    X_test_std = scaler.transform(X_all[test_idx]).astype(np.float32)

    FOLDS.append({
        "fold": fold,
        "train_idx": train_idx,
        "validation_idx": validation_idx,
        "test_idx": test_idx,
        "X_train_std": X_train_std,
        "X_validation_std": X_validation_std,
        "X_test_std": X_test_std,
    })

    total = len(X_all)
    print(
        f"Fold {fold}: "
        f"train={len(train_idx)} ({100*len(train_idx)/total:.2f}%), "
        f"validation={len(validation_idx)} ({100*len(validation_idx)/total:.2f}%), "
        f"test={len(test_idx)} ({100*len(test_idx)/total:.2f}%)"
    )

print("\n[OK] Los cinco folds están preparados.")
print("[OK] Validation y test tienen el mismo tamaño en cada fold.")
print("[OK] El escalado estándar se ajustó exclusivamente con train.")

Fold 1: train=12584 (59.99%), validation=4196 (20.00%), test=4196 (20.00%)
Fold 2: train=12586 (60.00%), validation=4195 (20.00%), test=4195 (20.00%)
Fold 3: train=12586 (60.00%), validation=4195 (20.00%), test=4195 (20.00%)
Fold 4: train=12586 (60.00%), validation=4195 (20.00%), test=4195 (20.00%)
Fold 5: train=12586 (60.00%), validation=4195 (20.00%), test=4195 (20.00%)

[OK] Los cinco folds están preparados.
[OK] Validation y test tienen el mismo tamaño en cada fold.
[OK] El escalado estándar se ajustó exclusivamente con train.


In [4]:
# ============================================================
# Celda 4 — Autoencoders acelerados
# ============================================================
class LinearAE(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int):
        super().__init__()
        self.encoder = nn.Linear(input_dim, latent_dim)
        self.decoder = nn.Linear(latent_dim, input_dim)

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def encode(self, x):
        return self.encoder(x)


class NonLinearAE(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int, hidden_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def encode(self, x):
        return self.encoder(x)


def make_autoencoder(ae_type: str, input_dim: int, latent_dim: int) -> nn.Module:
    if ae_type == "AE_lineal":
        return LinearAE(input_dim, latent_dim)
    if ae_type == "AE_no_lineal":
        return NonLinearAE(input_dim, latent_dim, AE_HIDDEN)
    raise ValueError(f"Autoencoder no reconocido: {ae_type}")


@torch.no_grad()
def encode_tensor_in_chunks(model, X_tensor, chunk_size=8192):
    model.eval()
    outputs = []

    for start in range(0, len(X_tensor), chunk_size):
        outputs.append(model.encode(X_tensor[start:start + chunk_size]).cpu())

    return torch.cat(outputs, dim=0).numpy().astype(np.float32)


def train_autoencoder_fast(
    X_train_std,
    X_validation_std,
    X_test_std,
    ae_type,
    latent_dim,
    fold,
):
    # Semilla distinta pero reproducible para cada combinación.
    type_offset = 0 if ae_type == "AE_lineal" else 100_000
    run_seed = SEED + type_offset + 100 * latent_dim + fold
    set_seed(run_seed)

    X_train_t = torch.as_tensor(
        X_train_std,
        dtype=torch.float32,
        device=DEVICE,
    )
    X_validation_t = torch.as_tensor(
        X_validation_std,
        dtype=torch.float32,
        device=DEVICE,
    )
    X_test_t = torch.as_tensor(
        X_test_std,
        dtype=torch.float32,
        device=DEVICE,
    )

    model = make_autoencoder(
        ae_type=ae_type,
        input_dim=X_train_std.shape[1],
        latent_dim=latent_dim,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=AE_LR,
        weight_decay=AE_WEIGHT_DECAY,
    )
    loss_fn = nn.MSELoss()

    best_state = None
    best_epoch = 0
    best_validation_mse = float("inf")
    epochs_without_improvement = 0
    history = []

    n_train = len(X_train_t)
    effective_batch = min(AE_BATCH_SIZE, n_train)

    for epoch in range(1, AE_MAX_EPOCHS + 1):
        model.train()

        # Barajado directamente en el dispositivo: evita DataLoader y transferencias repetidas.
        permutation = torch.randperm(n_train, device=DEVICE)
        sum_loss = 0.0
        n_seen = 0

        for start in range(0, n_train, effective_batch):
            idx = permutation[start:start + effective_batch]
            xb = X_train_t[idx]

            optimizer.zero_grad(set_to_none=True)
            reconstruction = model(xb)
            loss = loss_fn(reconstruction, xb)
            loss.backward()
            optimizer.step()

            batch_n = len(xb)
            sum_loss += float(loss.detach()) * batch_n
            n_seen += batch_n

        train_mse = sum_loss / max(n_seen, 1)

        model.eval()
        with torch.no_grad():
            validation_mse = float(
                loss_fn(model(X_validation_t), X_validation_t).detach()
            )

        history.append({
            "epoch": epoch,
            "train_mse": train_mse,
            "validation_mse": validation_mse,
        })

        if validation_mse < best_validation_mse - AE_MIN_DELTA:
            best_validation_mse = validation_mse
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= AE_PATIENCE:
            break

    if best_state is None:
        raise RuntimeError(
            f"No se obtuvo checkpoint para {ae_type}, k={latent_dim}, fold={fold}."
        )

    model.load_state_dict(best_state)
    model.to(DEVICE)
    model.eval()

    Z_train = encode_tensor_in_chunks(model, X_train_t)
    Z_validation = encode_tensor_in_chunks(model, X_validation_t)
    Z_test = encode_tensor_in_chunks(model, X_test_t)

    if SAVE_AE_WEIGHTS:
        torch.save(
            {
                "model_state_dict": best_state,
                "ae_type": ae_type,
                "input_dim": X_train_std.shape[1],
                "latent_dim": latent_dim,
                "fold": fold,
                "best_epoch": best_epoch,
                "best_validation_mse": best_validation_mse,
                "seed": run_seed,
            },
            WEIGHTS_DIR / f"{ae_type}_{latent_dim:02d}f_fold{fold}.pt",
        )

    # Liberación explícita.
    del model, X_train_t, X_validation_t, X_test_t
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return Z_train, Z_validation, Z_test, {
        "best_epoch": best_epoch,
        "best_validation_mse": best_validation_mse,
        "epochs_run": len(history),
    }

In [5]:
# ============================================================
# Celda 5 — Exportación segura y validación
# ============================================================
TRANSFORMS = ["PCA", "ICA", "AE_lineal", "AE_no_lineal"]


def scale_angles_train_only(Z_train, Z_validation, Z_test):
    scaler = MinMaxScaler(feature_range=(-SCALE, SCALE))

    train_scaled = scaler.fit_transform(Z_train).astype(np.float32)
    validation_scaled = scaler.transform(Z_validation).astype(np.float32)
    test_scaled = scaler.transform(Z_test).astype(np.float32)

    # Validation y test pueden exceder el rango aprendido en train.
    train_scaled = np.clip(train_scaled, -SCALE, SCALE)
    validation_scaled = np.clip(validation_scaled, -SCALE, SCALE)
    test_scaled = np.clip(test_scaled, -SCALE, SCALE)

    return train_scaled, validation_scaled, test_scaled


def build_split_dataframe(Z, indices, fold, split_name):
    if split_name not in {"train", "validation", "test"}:
        raise ValueError(f"split_name no válido: {split_name}")

    data = {
        "fold": np.full(len(indices), fold, dtype=np.int16),
        "split": np.full(len(indices), split_name, dtype=object),
        "sample_index": original_index_all[indices],
        "label": y_all[indices].astype(np.int64),
    }

    # Protección doble: ningún metadato puede sobrescribir columnas reservadas.
    for key, values in metadata_all.items():
        safe_key = "split_original" if key == "split" else key

        if safe_key in RESERVED_COLUMNS:
            raise RuntimeError(
                f"El metadato '{safe_key}' intenta sobrescribir una columna reservada."
            )

        data[safe_key] = np.asarray(values)[indices]

    for j in range(Z.shape[1]):
        data[f"feature_{j + 1:02d}"] = Z[:, j]

    df = pd.DataFrame(data)

    if df["split"].isna().any():
        raise RuntimeError("Se generaron valores vacíos en la nueva columna split.")
    if set(df["split"].unique()) != {split_name}:
        raise RuntimeError("La columna split fue alterada durante la exportación.")

    return df


def output_path(transform_name, n_features):
    return OUTPUT_DIR / f"{transform_name}_{n_features:02d}_caracteristicas.csv"


def expected_feature_columns(n_features):
    return [f"feature_{i:02d}" for i in range(1, n_features + 1)]


def validate_one_csv(path, transform_name, n_features, raise_on_error=False):
    errors = []

    try:
        df = pd.read_csv(path)
    except Exception as exc:
        errors.append(f"No se pudo leer: {exc}")
        df = None

    if df is not None:
        required = {
            "fold", "split", "sample_index", "label",
            *expected_feature_columns(n_features),
        }
        missing = required - set(df.columns)
        if missing:
            errors.append(f"Columnas faltantes: {sorted(missing)}")

        if "split" in df.columns:
            found_splits = set(df["split"].dropna().unique())
            expected_splits = {"train", "validation", "test"}
            if found_splits != expected_splits:
                errors.append(
                    f"Splits encontrados={sorted(found_splits)}; "
                    f"esperados={sorted(expected_splits)}"
                )
            if df["split"].isna().any():
                errors.append("La columna split contiene valores vacíos.")

        if "fold" in df.columns:
            found_folds = set(df["fold"].dropna().astype(int).unique())
            if found_folds != set(range(1, N_SPLITS + 1)):
                errors.append(f"Folds encontrados: {sorted(found_folds)}")

        if {"fold", "split", "sample_index"}.issubset(df.columns):
            for fold in range(1, N_SPLITS + 1):
                fold_df = df[df["fold"] == fold]
                counts = fold_df["split"].value_counts()

                n_val = int(counts.get("validation", 0))
                n_test = int(counts.get("test", 0))
                if n_val != n_test:
                    errors.append(
                        f"Fold {fold}: validation={n_val}, test={n_test}"
                    )

                split_sets = {
                    name: set(
                        fold_df.loc[
                            fold_df["split"] == name,
                            "sample_index",
                        ].tolist()
                    )
                    for name in ("train", "validation", "test")
                }

                if split_sets["train"] & split_sets["validation"]:
                    errors.append(f"Fold {fold}: solapamiento train-validation")
                if split_sets["train"] & split_sets["test"]:
                    errors.append(f"Fold {fold}: solapamiento train-test")
                if split_sets["validation"] & split_sets["test"]:
                    errors.append(f"Fold {fold}: solapamiento validation-test")

        feature_cols = [c for c in df.columns if c.startswith("feature_")]
        if len(feature_cols) != n_features:
            errors.append(
                f"Tiene {len(feature_cols)} features; se esperaban {n_features}."
            )
        elif not np.isfinite(df[feature_cols].to_numpy()).all():
            errors.append("Hay NaN o infinitos en las características.")

    valid = not errors

    if raise_on_error and errors:
        raise RuntimeError(f"{path.name}: " + " | ".join(errors))

    return valid, errors


def atomic_write_csv(df, destination):
    temporary = destination.with_suffix(".csv.tmp")
    df.to_csv(temporary, index=False)
    os.replace(temporary, destination)


def export_complete_csv(transform_name, n_features, fold_latents):
    frames = []

    for fold_data, latent_data in zip(FOLDS, fold_latents):
        fold = fold_data["fold"]

        Z_train, Z_validation, Z_test = scale_angles_train_only(
            latent_data["Z_train"],
            latent_data["Z_validation"],
            latent_data["Z_test"],
        )

        frames.extend([
            build_split_dataframe(
                Z_train,
                fold_data["train_idx"],
                fold,
                "train",
            ),
            build_split_dataframe(
                Z_validation,
                fold_data["validation_idx"],
                fold,
                "validation",
            ),
            build_split_dataframe(
                Z_test,
                fold_data["test_idx"],
                fold,
                "test",
            ),
        ])

    final_df = pd.concat(frames, ignore_index=True)
    destination = output_path(transform_name, n_features)
    atomic_write_csv(final_df, destination)

    validate_one_csv(
        destination,
        transform_name,
        n_features,
        raise_on_error=True,
    )

    return destination, len(final_df)

## Ejecución completa

La celda siguiente:

1. Genera primero los 15 CSV de PCA usando solo cinco ajustes PCA.
2. Genera los 15 CSV de ICA.
3. Entrena y exporta los 15 AE lineales.
4. Entrena y exporta los 15 AE no lineales.
5. Verifica cada CSV inmediatamente después de escribirlo.
6. Omite automáticamente un CSV que ya exista y sea válido.

Los autoencoders de cada dimensión siguen siendo modelos independientes:

```text
64 → 1 → 64
64 → 2 → 64
...
64 → 15 → 64
```

No se recorta un AE de 15 dimensiones para fabricar los demás.

In [6]:
# ============================================================
# Celda 6 — Generación de los 60 CSV
# ============================================================
run_summary = []
start_total = time.time()


def should_skip(transform_name, n_features):
    path = output_path(transform_name, n_features)

    if not (RESUME and path.exists()):
        return False

    valid, errors = validate_one_csv(
        path,
        transform_name,
        n_features,
        raise_on_error=False,
    )

    if valid:
        print(f"[OMITIDO] {path.name}: ya existe y está verificado.")
        return True

    print(f"[REGENERAR] {path.name}: {' | '.join(errors)}")
    return False


# ------------------------------------------------------------
# A) PCA — una sola descomposición de 15 componentes por fold
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("PCA")
print("=" * 90)

pca_cache = []

for fold_data in FOLDS:
    pca = PCA(
        n_components=MAX_FEATURES,
        svd_solver="full",
    )

    Z_train_15 = pca.fit_transform(
        fold_data["X_train_std"]
    ).astype(np.float32)

    Z_validation_15 = pca.transform(
        fold_data["X_validation_std"]
    ).astype(np.float32)

    Z_test_15 = pca.transform(
        fold_data["X_test_std"]
    ).astype(np.float32)

    pca_cache.append({
        "Z_train": Z_train_15,
        "Z_validation": Z_validation_15,
        "Z_test": Z_test_15,
        "explained_variance_ratio": pca.explained_variance_ratio_.copy(),
    })

for k in range(MIN_FEATURES, MAX_FEATURES + 1):
    if should_skip("PCA", k):
        continue

    start = time.time()
    fold_latents = [
        {
            "Z_train": item["Z_train"][:, :k],
            "Z_validation": item["Z_validation"][:, :k],
            "Z_test": item["Z_test"][:, :k],
        }
        for item in pca_cache
    ]

    path, n_rows = export_complete_csv("PCA", k, fold_latents)
    elapsed = time.time() - start

    run_summary.append({
        "transform": "PCA",
        "n_features": k,
        "rows": n_rows,
        "seconds": elapsed,
        "file": path.name,
    })
    print(f"[OK] {path.name} | {elapsed:.1f} s")


# ------------------------------------------------------------
# B) ICA — ajuste independiente para cada k y fold
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("ICA")
print("=" * 90)

for k in range(MIN_FEATURES, MAX_FEATURES + 1):
    if should_skip("ICA", k):
        continue

    start = time.time()
    fold_latents = []

    for fold_data in FOLDS:
        fold = fold_data["fold"]

        ica = FastICA(
            n_components=k,
            whiten="unit-variance",
            max_iter=2000,
            tol=1e-4,
            random_state=SEED + 100 * k + fold,
        )

        Z_train = ica.fit_transform(
            fold_data["X_train_std"]
        ).astype(np.float32)

        Z_validation = ica.transform(
            fold_data["X_validation_std"]
        ).astype(np.float32)

        Z_test = ica.transform(
            fold_data["X_test_std"]
        ).astype(np.float32)

        fold_latents.append({
            "Z_train": Z_train,
            "Z_validation": Z_validation,
            "Z_test": Z_test,
        })

    path, n_rows = export_complete_csv("ICA", k, fold_latents)
    elapsed = time.time() - start

    run_summary.append({
        "transform": "ICA",
        "n_features": k,
        "rows": n_rows,
        "seconds": elapsed,
        "file": path.name,
    })
    print(f"[OK] {path.name} | {elapsed:.1f} s")


# ------------------------------------------------------------
# C) Autoencoders — modelo independiente para cada k y fold
# ------------------------------------------------------------
for ae_type in ("AE_lineal", "AE_no_lineal"):
    print("\n" + "=" * 90)
    print(ae_type)
    print("=" * 90)

    for k in range(MIN_FEATURES, MAX_FEATURES + 1):
        if should_skip(ae_type, k):
            continue

        start = time.time()
        fold_latents = []
        ae_info_rows = []

        for fold_data in FOLDS:
            fold = fold_data["fold"]

            Z_train, Z_validation, Z_test, info = train_autoencoder_fast(
                X_train_std=fold_data["X_train_std"],
                X_validation_std=fold_data["X_validation_std"],
                X_test_std=fold_data["X_test_std"],
                ae_type=ae_type,
                latent_dim=k,
                fold=fold,
            )

            fold_latents.append({
                "Z_train": Z_train,
                "Z_validation": Z_validation,
                "Z_test": Z_test,
            })

            ae_info_rows.append({
                "transform": ae_type,
                "n_features": k,
                "fold": fold,
                **info,
            })

            print(
                f"  Fold {fold}: mejor época={info['best_epoch']}, "
                f"val MSE={info['best_validation_mse']:.8f}, "
                f"épocas ejecutadas={info['epochs_run']}"
            )

        path, n_rows = export_complete_csv(ae_type, k, fold_latents)

        pd.DataFrame(ae_info_rows).to_csv(
            TABLES_DIR / f"resumen_{ae_type}_{k:02d}f.csv",
            index=False,
        )

        elapsed = time.time() - start

        run_summary.append({
            "transform": ae_type,
            "n_features": k,
            "rows": n_rows,
            "seconds": elapsed,
            "file": path.name,
        })
        print(f"[OK] {path.name} | {elapsed/60:.1f} min")


summary_df = pd.DataFrame(run_summary)

if not summary_df.empty:
    summary_df.to_csv(
        TABLES_DIR / "resumen_ejecucion_actual.csv",
        index=False,
    )

print("\n" + "=" * 90)
print("GENERACIÓN TERMINADA")
print("=" * 90)
print(f"Tiempo de esta ejecución: {(time.time() - start_total) / 3600:.2f} h")
print(f"CSV presentes: {len(list(OUTPUT_DIR.glob('*.csv')))}")
print("Carpeta:", OUTPUT_DIR.resolve())


PCA
[OK] PCA_01_caracteristicas.csv | 0.8 s
[OK] PCA_02_caracteristicas.csv | 0.9 s
[OK] PCA_03_caracteristicas.csv | 1.1 s
[OK] PCA_04_caracteristicas.csv | 1.2 s
[OK] PCA_05_caracteristicas.csv | 1.3 s
[OK] PCA_06_caracteristicas.csv | 1.5 s
[OK] PCA_07_caracteristicas.csv | 1.6 s
[OK] PCA_08_caracteristicas.csv | 1.8 s
[OK] PCA_09_caracteristicas.csv | 2.0 s
[OK] PCA_10_caracteristicas.csv | 2.1 s
[OK] PCA_11_caracteristicas.csv | 2.4 s
[OK] PCA_12_caracteristicas.csv | 2.4 s
[OK] PCA_13_caracteristicas.csv | 2.7 s
[OK] PCA_14_caracteristicas.csv | 2.9 s
[OK] PCA_15_caracteristicas.csv | 3.1 s

ICA
[OK] ICA_01_caracteristicas.csv | 2.2 s
[OK] ICA_02_caracteristicas.csv | 2.4 s
[OK] ICA_03_caracteristicas.csv | 2.5 s
[OK] ICA_04_caracteristicas.csv | 2.6 s
[OK] ICA_05_caracteristicas.csv | 2.9 s
[OK] ICA_06_caracteristicas.csv | 3.1 s
[OK] ICA_07_caracteristicas.csv | 3.3 s
[OK] ICA_08_caracteristicas.csv | 3.4 s
[OK] ICA_09_caracteristicas.csv | 3.6 s
[OK] ICA_10_caracteristicas.cs

In [7]:
# ============================================================
# Celda 7 — Verificación final estricta de los 60 CSV
# ============================================================
expected_files = {
    f"{transform}_{k:02d}_caracteristicas.csv"
    for transform in TRANSFORMS
    for k in range(MIN_FEATURES, MAX_FEATURES + 1)
}
actual_files = {path.name for path in OUTPUT_DIR.glob("*.csv")}

missing_files = sorted(expected_files - actual_files)
unexpected_files = sorted(actual_files - expected_files)

print("CSV encontrados:", len(actual_files))
print("Esperados:", len(expected_files))
print("Faltantes:", missing_files or "ninguno")
print("No esperados:", unexpected_files or "ninguno")

all_errors = []

for transform in TRANSFORMS:
    for k in range(MIN_FEATURES, MAX_FEATURES + 1):
        path = output_path(transform, k)

        if not path.exists():
            all_errors.append(f"{path.name}: no existe")
            continue

        valid, errors = validate_one_csv(
            path,
            transform,
            k,
            raise_on_error=False,
        )

        if not valid:
            all_errors.extend(
                [f"{path.name}: {error}" for error in errors]
            )

if missing_files:
    all_errors.extend([f"Falta: {name}" for name in missing_files])

if unexpected_files:
    all_errors.extend([f"No esperado: {name}" for name in unexpected_files])

if all_errors:
    print("\nERRORES:")
    for error in all_errors[:100]:
        print("-", error)
    raise RuntimeError(
        f"La verificación final encontró {len(all_errors)} errores."
    )

# Comprobación visual de la columna split en un archivo AE.
example_check = pd.read_csv(
    OUTPUT_DIR / "AE_lineal_04_caracteristicas.csv"
)

print("\nConteos de ejemplo: AE_lineal_04_caracteristicas.csv")
print(
    example_check.groupby(["fold", "split"])
    .size()
    .unstack(fill_value=0)
)

print("\nValores únicos de la nueva columna split:")
print(example_check["split"].unique())

if "split_original" in example_check.columns:
    print("\nEl split antiguo se conserva aparte como split_original.")

print("\n[OK] Los 60 CSV están completos.")
print("[OK] La columna split solo contiene train, validation y test.")
print("[OK] No hay valores vacíos en split.")
print("[OK] Validation y test tienen el mismo tamaño en todos los folds.")
print("[OK] No hay muestras compartidas entre splits dentro de un fold.")

CSV encontrados: 60
Esperados: 60
Faltantes: ninguno
No esperados: ninguno

Conteos de ejemplo: AE_lineal_04_caracteristicas.csv
split  test  train  validation
fold                          
1      4196  12584        4196
2      4195  12586        4195
3      4195  12586        4195
4      4195  12586        4195
5      4195  12586        4195

Valores únicos de la nueva columna split:
['train' 'validation' 'test']

El split antiguo se conserva aparte como split_original.

[OK] Los 60 CSV están completos.
[OK] La columna split solo contiene train, validation y test.
[OK] No hay valores vacíos en split.
[OK] Validation y test tienen el mismo tamaño en todos los folds.
[OK] No hay muestras compartidas entre splits dentro de un fold.


In [8]:
# ============================================================
# Celda 8 — Ejemplo listo para Camila
# ============================================================
TRANSFORMACION = "AE_lineal"
NUM_CARACTERISTICAS = 4
FOLD = 1

camila_path = OUTPUT_DIR / (
    f"{TRANSFORMACION}_{NUM_CARACTERISTICAS:02d}_caracteristicas.csv"
)

camila_df = pd.read_csv(camila_path)

feature_columns = [
    f"feature_{i:02d}"
    for i in range(1, NUM_CARACTERISTICAS + 1)
]

train_df = camila_df[
    (camila_df["fold"] == FOLD)
    & (camila_df["split"] == "train")
].copy()

validation_df = camila_df[
    (camila_df["fold"] == FOLD)
    & (camila_df["split"] == "validation")
].copy()

test_df = camila_df[
    (camila_df["fold"] == FOLD)
    & (camila_df["split"] == "test")
].copy()

X_train = train_df[feature_columns].to_numpy(dtype=np.float32)
y_train = train_df["label"].to_numpy(dtype=np.int64)

X_validation = validation_df[feature_columns].to_numpy(dtype=np.float32)
y_validation = validation_df["label"].to_numpy(dtype=np.int64)

X_test = test_df[feature_columns].to_numpy(dtype=np.float32)
y_test = test_df["label"].to_numpy(dtype=np.int64)

print("Archivo:", camila_path)
print("X_train      :", X_train.shape, "| y_train      :", y_train.shape)
print("X_validation :", X_validation.shape, "| y_validation :", y_validation.shape)
print("X_test       :", X_test.shape, "| y_test       :", y_test.shape)

assert len(X_validation) == len(X_test)
assert len(X_train) > 0
assert len(X_validation) > 0
assert len(X_test) > 0

print("\n[OK] Datos listos para Camila.")

Archivo: Comparacion_de_transformaciones_de_1_a_15_caracteristicas_con_validacion_60_20_20\AE_lineal_04_caracteristicas.csv
X_train      : (12584, 4) | y_train      : (12584,)
X_validation : (4196, 4) | y_validation : (4196,)
X_test       : (4196, 4) | y_test       : (4196,)

[OK] Datos listos para Camila.
